# D161 — Importing Olist CSV Data into MySQL

MySQL runs inside WSL with Ubuntu 24.04. The Olist CSV files are stored on the Windows C drive. WSL sees the same directory here:

```text
Windows: C:\\data\\olist
WSL:     /mnt/c/data/olist
```

The customers file is imported first as a simple test. The notebook then creates the full relational schema and imports all nine Olist CSV files.

## 1. Check the files in Ubuntu

Open the Ubuntu terminal and move to the data directory. The prompt below is only an example. Type the command after `$`.

```bash
user@ubuntu:~$ cd /mnt/c/data/olist
user@ubuntu:/mnt/c/data/olist$ ls
```

The folder contains nine CSV files.

## 2. Show the first and last five records

A CSV file has one header line. `head -n 6` shows the header and the first five data records. `tail -n 5` shows the last five data records.

### Customers

```bash
head -n 6 olist_customers_dataset.csv
tail -n 5 olist_customers_dataset.csv
```

### Geolocation

```bash
head -n 6 olist_geolocation_dataset.csv
tail -n 5 olist_geolocation_dataset.csv
```

### Orders

```bash
head -n 6 olist_orders_dataset.csv
tail -n 5 olist_orders_dataset.csv
```

### Order items

```bash
head -n 6 olist_order_items_dataset.csv
tail -n 5 olist_order_items_dataset.csv
```

### Payments

```bash
head -n 6 olist_order_payments_dataset.csv
tail -n 5 olist_order_payments_dataset.csv
```

### Reviews

```bash
head -n 6 olist_order_reviews_dataset.csv
tail -n 5 olist_order_reviews_dataset.csv
```

### Products

```bash
head -n 6 olist_products_dataset.csv
tail -n 5 olist_products_dataset.csv
```

### Sellers

```bash
head -n 6 olist_sellers_dataset.csv
tail -n 5 olist_sellers_dataset.csv
```

### Category translation

```bash
head -n 6 product_category_name_translation.csv
tail -n 5 product_category_name_translation.csv
```

### A shorter command for all nine files

The loop below performs the same check for every CSV file.

```bash
for file in *.csv; do
    echo "===== $file : first 5 records ====="
    head -n 6 "$file"
    echo "===== $file : last 5 records ====="
    tail -n 5 "$file"
    echo
done
```

## 3. `INFILE` and `LOCAL INFILE`

MySQL provides two related commands.

| Command | Who reads the file? | Path belongs to | Setting |
|---|---|---|---|
| `LOAD DATA INFILE` | MySQL server | Ubuntu server | restricted by `secure_file_priv` |
| `LOAD DATA LOCAL INFILE` | Python/MySQL client | computer running the notebook | server `local_infile=ON` and client `allow_local_infile=True` |

The live server reports `secure_file_priv=/var/lib/mysql-files/`. Therefore a server-side `LOAD DATA INFILE` cannot read `/mnt/c/data/olist` directly. Copying files into `/var/lib/mysql-files/` would also require `sudo` permission.

This notebook uses **`LOAD DATA LOCAL INFILE`**. Python reads `C:/data/olist/olist_customers_dataset.csv` and sends it to MySQL in WSL. This is still a bulk MySQL import; Python does not insert the rows one at a time.

## 4. Enable local file loading in MySQL

Check the current server setting from the Ubuntu terminal:

```bash
mysql -u root -p -e "SHOW VARIABLES LIKE 'local_infile';"
```

Enter the MySQL password when prompted. If the value is `OFF`, enable it:

```bash
mysql -u root -p -e "SET GLOBAL local_infile = ON;"
mysql -u root -p -e "SHOW VARIABLES LIKE 'local_infile';"
```

`SET GLOBAL` remains active until MySQL restarts. To keep the setting after restart, add this under `[mysqld]` in the MySQL server configuration and restart MySQL:

```ini
[mysqld]
local_infile=1
```

A common Ubuntu configuration file is `/etc/mysql/mysql.conf.d/mysqld.cnf`. Changing it requires administrator access. The setting was enabled successfully on the test server for this lesson.

## 5. Python package and environment variables

The notebook uses `mysql-connector-python`. Install it once if it is missing:

```bash
pip install mysql-connector-python
```

The connection reads these environment variables:

- `MYSQL_HOSTNAME` — defaults to `127.0.0.1`
- `MYSQL_PORT` — defaults to `3306`
- `MYSQL_USERNAME` — defaults to `root`
- `MYSQL_PASSWORD` — defaults to `root` for this local class environment
- `MYSQL_DATABASE` — defaults to `olist_import_lab`

Environment variables are preferred because a real password should not be saved in a notebook. The `root/root` defaults are only for this local learning setup.

In [ ]:
import os
from pathlib import Path
import mysql.connector
from mysql.connector import Error

MYSQL_HOSTNAME = os.environ.get("MYSQL_HOSTNAME", "127.0.0.1")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", "3306"))
MYSQL_USERNAME = os.environ.get("MYSQL_USERNAME", "root")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "root")
MYSQL_DATABASE = os.environ.get("MYSQL_DATABASE", "olist_import_lab")
CSV_PATH = Path(r"C:\data\olist\olist_customers_dataset.csv")

assert CSV_PATH.is_file(), f"CSV file not found: {CSV_PATH}"
print("CSV file:", CSV_PATH)

## 6. Connect and create the database

The first connection does not select a database because the database may not exist yet. `allow_local_infile=True` is required in the Python client. The server setting alone is not enough for `LOAD DATA LOCAL INFILE`.

In [ ]:
server_connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
    allow_local_infile=True,
)
server_cursor = server_connection.cursor()
server_cursor.execute(
    f"CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}` "
    "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
)
server_cursor.close()
server_connection.close()
print(f"Database {MYSQL_DATABASE!r} is ready.")

In [ ]:
connection = mysql.connector.connect(
    host=MYSQL_HOSTNAME,
    port=MYSQL_PORT,
    user=MYSQL_USERNAME,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
    allow_local_infile=True,
)
print("Connected:", connection.is_connected())
print("MySQL version:", connection.server_info)

## 7. SQL helper with formatted results

A cursor sends SQL to MySQL and reads the returned rows. `execute_sql` prints query results as an aligned table. Statements that change data are committed. If a statement fails, the transaction is rolled back and the original error is shown.

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row] for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    line = "-+-".join("-" * width for width in widths)
    print(" | ".join(str(column).ljust(width) for column, width in zip(columns, widths)))
    print(line)
    for row in text_rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))


def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [item[0] for item in cursor.description]
            rows = cursor.fetchall()
            print_rows(columns, rows)
            return rows

        affected_rows = cursor.rowcount
        connection.commit()
        print(f"Statement completed. Affected rows: {affected_rows:,}")
        return affected_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## 8. Check the import settings from Python

`local_infile` must show `ON`. `secure_file_priv` is shown only to explain why direct server-side loading from `/mnt/c` is not used here.

In [ ]:
execute_sql("SHOW VARIABLES LIKE 'local_infile'")
execute_sql("SHOW VARIABLES LIKE 'secure_file_priv'")

## 9. Create the customers table

The table matches the five CSV columns.

- The two IDs contain 32 characters, so `CHAR(32)` is used.
- ZIP prefix uses `CHAR(5)`. This preserves values such as `06273`; an integer would remove the first zero.
- `customer_id` is the primary key.
- `customer_unique_id` is not unique because one buyer can place several orders.
- The extra indexes support common searches by buyer, ZIP prefix, state, and city.

In [ ]:
create_customers_sql = """
CREATE TABLE IF NOT EXISTS olist_customers (
    customer_id CHAR(32) NOT NULL,
    customer_unique_id CHAR(32) NOT NULL,
    customer_zip_code_prefix CHAR(5) NOT NULL,
    customer_city VARCHAR(100) NOT NULL,
    customer_state CHAR(2) NOT NULL,

    CONSTRAINT pk_olist_customers PRIMARY KEY (customer_id),
    INDEX idx_customers_unique_id (customer_unique_id),
    INDEX idx_customers_zip (customer_zip_code_prefix),
    INDEX idx_customers_state_city (customer_state, customer_city)
) ENGINE=InnoDB
  DEFAULT CHARACTER SET utf8mb4
  COLLATE utf8mb4_unicode_ci
"""
execute_sql(create_customers_sql)
execute_sql("DESCRIBE olist_customers")

## 10. Import the customers CSV

The import uses the Windows path because the Python notebook is the client reading the file. Forward slashes work well in a MySQL string.

The file uses a newline at the end of each record. On some Windows-created CSVs, the final field can contain a carriage-return character (`\r`). The user variable `@customer_state` and `TRIM` remove it safely.

The cell loads data only when the table is empty. This prevents a second run from producing primary-key errors.

In [ ]:
existing_rows = execute_sql("SELECT COUNT(*) AS row_count FROM olist_customers")[0][0]

if existing_rows == 0:
    csv_for_mysql = CSV_PATH.as_posix()
    load_customers_sql = f"""
    LOAD DATA LOCAL INFILE '{csv_for_mysql}'
    INTO TABLE olist_customers
    CHARACTER SET utf8mb4
    FIELDS TERMINATED BY ','
           OPTIONALLY ENCLOSED BY '"'
    LINES TERMINATED BY '\n'
    IGNORE 1 LINES
    (customer_id, customer_unique_id, customer_zip_code_prefix,
     customer_city, @customer_state)
    SET customer_state = TRIM(TRAILING '\r' FROM @customer_state)
    """
    execute_sql(load_customers_sql)
else:
    print(f"Import skipped: olist_customers already contains {existing_rows:,} rows.")

## 11. Check the imported data

The source file contains 99,441 records. `customer_id` must also have 99,441 distinct values. `customer_unique_id` has fewer distinct values because some buyers placed more than one order.

In [ ]:
validation_sql = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS distinct_customer_ids,
    COUNT(DISTINCT customer_unique_id) AS distinct_buyers,
    SUM(customer_id IS NULL) AS null_customer_ids
FROM olist_customers
"""
execute_sql(validation_sql)

In [ ]:
execute_sql("""
SELECT *
FROM olist_customers
ORDER BY customer_id
LIMIT 5
""")

In [ ]:
execute_sql("""
SELECT *
FROM olist_customers
ORDER BY customer_id DESC
LIMIT 5
""")

### Tested result

This workflow was tested against the local MySQL server in WSL. The result was:

| Check | Result |
|---|---:|
| Imported rows | 99,441 |
| Distinct `customer_id` | 99,441 |
| Distinct `customer_unique_id` | 96,096 |

A ZIP value such as `06273` remained five characters, confirming that leading zeros were preserved.

## 12. Useful checks and common errors

| Message or problem | Meaning | Fix |
|---|---|---|
| Loading local data is disabled | server setting is off | run `SET GLOBAL local_infile = ON` as an administrator |
| LOCAL INFILE request rejected | Python client did not opt in | reconnect with `allow_local_infile=True` |
| File not found | Python cannot see the path | check `C:/data/olist/olist_customers_dataset.csv` |
| Duplicate primary key | data was loaded before | keep the empty-table check or use a new table |
| State contains an extra character | CSV uses CRLF line endings | keep the `@customer_state` and `TRIM` expression |
| ZIP has fewer than five digits | ZIP was treated as a number | store it as `CHAR(5)` |
| Access denied for `LOAD DATA INFILE` | server path is outside `secure_file_priv` | use `LOCAL`, or copy the file to the permitted server directory |

Check warnings immediately after an import if the row count is unexpected:

```python
execute_sql("SHOW WARNINGS LIMIT 20")
```

## 13. Create the remaining eight tables

Parent tables are created before child tables so that the foreign keys can be added. Geolocation uses a generated primary key because ZIP prefixes repeat. Reviews use (`review_id`, `order_id`) because that pair is unique in the source.

In [ ]:
remaining_table_statements = [
"""CREATE TABLE IF NOT EXISTS olist_geolocation (
 geolocation_id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
 geolocation_zip_code_prefix CHAR(5) NOT NULL, geolocation_lat DECIMAL(10,7) NOT NULL,
 geolocation_lng DECIMAL(10,7) NOT NULL, geolocation_city VARCHAR(100) NOT NULL,
 geolocation_state CHAR(2) NOT NULL, PRIMARY KEY (geolocation_id),
 INDEX idx_geo_zip (geolocation_zip_code_prefix),
 INDEX idx_geo_state_city (geolocation_state, geolocation_city)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_sellers (
 seller_id CHAR(32) NOT NULL, seller_zip_code_prefix CHAR(5) NOT NULL,
 seller_city VARCHAR(100) NOT NULL, seller_state CHAR(2) NOT NULL, PRIMARY KEY (seller_id),
 INDEX idx_seller_zip (seller_zip_code_prefix),
 INDEX idx_seller_state_city (seller_state, seller_city)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS product_category_translation (
 product_category_name VARCHAR(100) NOT NULL, product_category_name_english VARCHAR(100) NULL,
 PRIMARY KEY (product_category_name),
 UNIQUE KEY uq_category_english (product_category_name_english)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_products (
 product_id CHAR(32) NOT NULL, product_category_name VARCHAR(100) NULL,
 product_name_lenght SMALLINT UNSIGNED NULL, product_description_lenght INT UNSIGNED NULL,
 product_photos_qty SMALLINT UNSIGNED NULL, product_weight_g DECIMAL(10,2) NULL,
 product_length_cm DECIMAL(10,2) NULL, product_height_cm DECIMAL(10,2) NULL,
 product_width_cm DECIMAL(10,2) NULL, PRIMARY KEY (product_id),
 INDEX idx_product_category (product_category_name),
 CONSTRAINT fk_product_category FOREIGN KEY (product_category_name)
   REFERENCES product_category_translation (product_category_name)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_orders (
 order_id CHAR(32) NOT NULL, customer_id CHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL,
 order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL,
 order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL,
 order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id),
 UNIQUE KEY uq_orders_customer (customer_id),
 INDEX idx_orders_status_purchase (order_status, order_purchase_timestamp),
 CONSTRAINT fk_order_customer FOREIGN KEY (customer_id) REFERENCES olist_customers (customer_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_order_items (
 order_id CHAR(32) NOT NULL, order_item_id SMALLINT UNSIGNED NOT NULL,
 product_id CHAR(32) NOT NULL, seller_id CHAR(32) NOT NULL,
 shipping_limit_date DATETIME NOT NULL, price DECIMAL(12,2) NOT NULL,
 freight_value DECIMAL(12,2) NOT NULL, PRIMARY KEY (order_id, order_item_id),
 INDEX idx_item_product (product_id), INDEX idx_item_seller (seller_id),
 CONSTRAINT fk_item_order FOREIGN KEY (order_id) REFERENCES olist_orders (order_id),
 CONSTRAINT fk_item_product FOREIGN KEY (product_id) REFERENCES olist_products (product_id),
 CONSTRAINT fk_item_seller FOREIGN KEY (seller_id) REFERENCES olist_sellers (seller_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_order_payments (
 order_id CHAR(32) NOT NULL, payment_sequential SMALLINT UNSIGNED NOT NULL,
 payment_type VARCHAR(20) NOT NULL, payment_installments SMALLINT UNSIGNED NOT NULL,
 payment_value DECIMAL(12,2) NOT NULL, PRIMARY KEY (order_id, payment_sequential),
 INDEX idx_payment_type (payment_type),
 CONSTRAINT fk_payment_order FOREIGN KEY (order_id) REFERENCES olist_orders (order_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci""",
"""CREATE TABLE IF NOT EXISTS olist_order_reviews (
 review_id CHAR(32) NOT NULL, order_id CHAR(32) NOT NULL, review_score TINYINT UNSIGNED NOT NULL,
 review_comment_title VARCHAR(255) NULL, review_comment_message TEXT NULL,
 review_creation_date DATETIME NOT NULL, review_answer_timestamp DATETIME NOT NULL,
 PRIMARY KEY (review_id, order_id), INDEX idx_review_order (order_id),
 INDEX idx_review_score (review_score),
 CONSTRAINT chk_review_score CHECK (review_score BETWEEN 1 AND 5),
 CONSTRAINT fk_review_order FOREIGN KEY (order_id) REFERENCES olist_orders (order_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci"""
]

for statement in remaining_table_statements:
    execute_sql(statement)

## 14. Import the remaining files

The files are loaded in dependency order. Blank values become SQL `NULL`, and text dates become MySQL `DATETIME` values.

Two categories used by products are missing from the translation CSV. Their Portuguese keys are added with a null English name. This allows the product category foreign key to remain enforced without inventing translations.

One review comment ends with a literal backslash. `ESCAPED BY ''` keeps it as ordinary text. Without that clause, MySQL joins two CSV records and loads one review too few.

In [ ]:
DATA_DIR = CSV_PATH.parent
load_statements = {
"olist_geolocation": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_geolocation_dataset.csv').as_posix()}'
 INTO TABLE olist_geolocation CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, @state)
 SET geolocation_state=TRIM(TRAILING '\r' FROM @state)""",
"olist_sellers": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_sellers_dataset.csv').as_posix()}'
 INTO TABLE olist_sellers CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (seller_id, seller_zip_code_prefix, seller_city, @state)
 SET seller_state=TRIM(TRAILING '\r' FROM @state)""",
"product_category_translation": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'product_category_name_translation.csv').as_posix()}'
 INTO TABLE product_category_translation CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (product_category_name, @english)
 SET product_category_name_english=NULLIF(TRIM(TRAILING '\r' FROM @english),'')""",
"olist_products": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_products_dataset.csv').as_posix()}'
 INTO TABLE olist_products CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (product_id,@category,@name_len,@desc_len,@photos,@weight,@length,@height,@width)
 SET product_category_name=NULLIF(@category,''),
 product_name_lenght=NULLIF(@name_len,''), product_description_lenght=NULLIF(@desc_len,''),
 product_photos_qty=NULLIF(@photos,''), product_weight_g=NULLIF(@weight,''),
 product_length_cm=NULLIF(@length,''), product_height_cm=NULLIF(@height,''),
 product_width_cm=NULLIF(TRIM(TRAILING '\r' FROM @width),'')""",
"olist_orders": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_orders_dataset.csv').as_posix()}'
 INTO TABLE olist_orders CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (order_id,customer_id,order_status,@purchase,@approved,@carrier,@delivered,@estimated)
 SET order_purchase_timestamp=STR_TO_DATE(@purchase,'%Y-%m-%d %H:%i:%s'),
 order_approved_at=STR_TO_DATE(NULLIF(@approved,''),'%Y-%m-%d %H:%i:%s'),
 order_delivered_carrier_date=STR_TO_DATE(NULLIF(@carrier,''),'%Y-%m-%d %H:%i:%s'),
 order_delivered_customer_date=STR_TO_DATE(NULLIF(@delivered,''),'%Y-%m-%d %H:%i:%s'),
 order_estimated_delivery_date=STR_TO_DATE(TRIM(TRAILING '\r' FROM @estimated),'%Y-%m-%d %H:%i:%s')""",
"olist_order_items": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_order_items_dataset.csv').as_posix()}'
 INTO TABLE olist_order_items CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (order_id,order_item_id,product_id,seller_id,@shipping,price,@freight)
 SET shipping_limit_date=STR_TO_DATE(@shipping,'%Y-%m-%d %H:%i:%s'),
 freight_value=TRIM(TRAILING '\r' FROM @freight)""",
"olist_order_payments": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_order_payments_dataset.csv').as_posix()}'
 INTO TABLE olist_order_payments CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' LINES TERMINATED BY '\n' IGNORE 1 LINES
 (order_id,payment_sequential,payment_type,payment_installments,@value)
 SET payment_value=TRIM(TRAILING '\r' FROM @value)""",
"olist_order_reviews": f"""LOAD DATA LOCAL INFILE '{(DATA_DIR / 'olist_order_reviews_dataset.csv').as_posix()}'
 INTO TABLE olist_order_reviews CHARACTER SET utf8mb4
 FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' ESCAPED BY ''
 LINES TERMINATED BY '\n' IGNORE 1 LINES
 (review_id,order_id,review_score,@title,@message,@created,@answered)
 SET review_comment_title=NULLIF(@title,''), review_comment_message=NULLIF(@message,''),
 review_creation_date=STR_TO_DATE(@created,'%Y-%m-%d %H:%i:%s'),
 review_answer_timestamp=STR_TO_DATE(TRIM(TRAILING '\r' FROM @answered),'%Y-%m-%d %H:%i:%s')"""
}

for table_name, load_sql in load_statements.items():
    current_count = execute_sql(f"SELECT COUNT(*) FROM `{table_name}`")[0][0]
    if current_count == 0:
        print(f"Loading {table_name} ...")
        execute_sql(load_sql)
    else:
        print(f"Skipping {table_name}: already has {current_count:,} rows.")

    if table_name == 'product_category_translation':
        execute_sql("""INSERT IGNORE INTO product_category_translation
          (product_category_name, product_category_name_english) VALUES
          ('pc_gamer', NULL),
          ('portateis_cozinha_e_preparadores_de_alimentos', NULL)""")

## 15. Validate all table counts

The translation table has 73 rows after adding the two missing category keys. Every other expected count is the number of records in its CSV file.

In [ ]:
expected_counts = {
 'olist_customers': 99_441, 'olist_geolocation': 1_000_163,
 'olist_orders': 99_441, 'olist_order_items': 112_650,
 'olist_order_payments': 103_886, 'olist_order_reviews': 99_224,
 'olist_products': 32_951, 'olist_sellers': 3_095,
 'product_category_translation': 73,
}
parts = [f"SELECT '{t}' table_name, COUNT(*) actual, {n} expected FROM `{t}`"
         for t, n in expected_counts.items()]
count_results = execute_sql(' UNION ALL '.join(parts))
assert all(actual == expected for _, actual, expected in count_results)
print('All row counts match.')

## 16. View the keys and relationships

The first query lists primary and unique constraints. The second lists the seven foreign keys enforced by MySQL.

In [ ]:
execute_sql("""SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE constraint_schema=%s AND constraint_type IN ('PRIMARY KEY','UNIQUE')
ORDER BY table_name, constraint_type, constraint_name""", (MYSQL_DATABASE,))

In [ ]:
execute_sql("""SELECT table_name, column_name, constraint_name,
       referenced_table_name, referenced_column_name
FROM information_schema.key_column_usage
WHERE constraint_schema=%s AND referenced_table_name IS NOT NULL
ORDER BY table_name, constraint_name, ordinal_position""", (MYSQL_DATABASE,))

## 17. Check for missing parent records

Every result should be zero. Foreign keys already enforce these rules, but this query makes the checks visible.

In [ ]:
execute_sql("""
SELECT 'orders without customer' check_name, COUNT(*) problem_rows
FROM olist_orders o LEFT JOIN olist_customers c ON c.customer_id=o.customer_id
WHERE c.customer_id IS NULL
UNION ALL SELECT 'items without order', COUNT(*) FROM olist_order_items i
LEFT JOIN olist_orders o ON o.order_id=i.order_id WHERE o.order_id IS NULL
UNION ALL SELECT 'items without product', COUNT(*) FROM olist_order_items i
LEFT JOIN olist_products p ON p.product_id=i.product_id WHERE p.product_id IS NULL
UNION ALL SELECT 'items without seller', COUNT(*) FROM olist_order_items i
LEFT JOIN olist_sellers s ON s.seller_id=i.seller_id WHERE s.seller_id IS NULL
UNION ALL SELECT 'payments without order', COUNT(*) FROM olist_order_payments p
LEFT JOIN olist_orders o ON o.order_id=p.order_id WHERE o.order_id IS NULL
UNION ALL SELECT 'reviews without order', COUNT(*) FROM olist_order_reviews r
LEFT JOIN olist_orders o ON o.order_id=r.order_id WHERE o.order_id IS NULL
""")

## 18. Preview every imported table

This prints five MySQL rows from each table.

In [ ]:
for table_name in expected_counts:
    print(f'\n===== {table_name} =====')
    execute_sql(f"SELECT * FROM `{table_name}` LIMIT 5")

## 19. Tested result

The complete import was tested with MySQL running in Ubuntu 24.04 under WSL. All nine CSV files loaded with their expected counts. The database has seven active foreign keys, and all 99,224 reviews loaded correctly.

## 20. Close the connection

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed.')